<a href="https://colab.research.google.com/github/ajcollins16/DSST289_Fall2026_AJC/blob/main/Taylor_notebooks/notebook11a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook11a

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
metro = pl.read_csv(ub + "data/acs_cbsa.csv")
docs = pl.read_csv(ub + "data/wiki_senator_class_docs.csv")
meta = pl.read_csv(ub + "data/wiki_senator_class.csv")
anno = pl.read_csv(ub + "data/wiki_senator_class_anno.csv")

## Part I: Classify Household Income

1. Start by building a model object that uses linear regression to predict the household median income using the 1 bedroom median rent. Save the model as an object `model`.

2. Compute the score of the model object. This is easy, but take a moment to make sure you understand what this means.

3. Take the `metro` data and direclty comput the root mean squared error (RMSE) without using the `.score()` method. Verify that it matches what you had in the previous question.

4. Print out the coefficents from the model. Make sure you understand what these mean and how they relate to the notes from today.

5. Redefine `model` to be a linear regression predicting household median income using three variables: 1BR median rent, the percentage of income used for rent, and the percentage of people who own their homes.

6. Compute the score of this model (you may use `.score()` again). Visually compare to the model with just one predictor.

7. Look at the coefficents of the new model. What do their signs tell us about the relationship between these three predictor variables and the household median income?

8. Next, redefine `model`. We will still predict household median income using 1BR median rent, percentage spent on housing, and percentage of people owning their house. But, now use the `DSSklearn.elastic_net_cv` model.

9. Look at the coefficents of the elastic net model. You should see that they are much smaller in magnitude than the logistic regression model.

10. Look at the score of the model. It's not very good (probably) because the cross-validation system is not optimized for models this small. We would need to modify some inputs to get better values.

## Part II: Classify Region

11. Now, use logistic regression to predict the quadrant (`quad`) that a region is from using the density, percentage owned, and household median income. Save the model again as `model`.

12. Now look at the coefficents of the model. Try your best to make some sense of them.

13. Compute the scores for this model. Make sure you understand what these numbers represent! Hint: It's easier to understand than in the previous examples.

14. Print out the confusion matrix. Make sure you also understand what it represents.

## Part III: Text Analysis

15. Let's load in the annotation data from the sentator dataset we built previously. (This code and the remaining ones I have already filled in for you)

In [ ]:
anno = (
    anno
    .join(meta.select(c.doc_id, c.party), on=c.doc_id)
)
anno

16. We can fit a penalised logistic model using the term-frequency matrix by calling the function `DSSklearnText.logistic_regression_cv`. It works just like the logistic regression model above, but accepts an annotation object and does the creation of the TF matrix internally for us. We will do more with these in future notebooks.

In [ ]:
model =(
    anno
    .filter(c.upos.is_in(["ADJ"]))
    .pipe(
        DSSklearnText.logistic_regression_cv,
        doc_id=c.doc_id,
        term_id=c.lemma,
        target=c.party,
        solver="saga",
        min_df=0.05,
        l1_ratios=[1]
    )
)

17. Look at the confusion matrix:

In [ ]:
model.confusion_matrix()


18. Finally, look at the non-zero coefficents. Notice that the table has a very specific (where Republican values are all the negative of the Democratic ones) due to the way the data is balanced between the categories.

In [ ]:
(
    model
    .coef(scaled=True)
    .with_columns(cs = c.Democratic.abs() + c.Republican.abs())
    .filter(c.cs > 0)
    .sort(c.Democratic)
)